# Inspect handwriting-aware tokenizers

Run the cells in order. Edit **`TEXT`** and **`TOKENIZER_NAME`** in section 1 to inspect segmentation, token IDs and decoded text. Section 3 accepts your own frame IDs or model scores and uses TVA's actual `BestPath` decoder.

The default demonstration uses the English ED tokenizers. The historical OnHW Bigram is also selectable; its alphabet supports words and differs from ED's sentence alphabet. This notebook needs only TVA and the bundled frozen tokenizer files—no dataset, checkpoint or GPU. It loads vocabularies and runs CPU decoding; it does not train anything.

Handwriting evidence influences vocabulary construction. Encoding new text uses the frozen vocabulary and segmentation rules; it does not read handwriting images.


In [1]:
from pathlib import Path
import sys
from time import perf_counter

_startup = perf_counter()
print("Starting tokenizer inspection", flush=True)
print("Python:", sys.executable, flush=True)
import hashlib
import unicodedata

print("Importing PyTorch...", flush=True)
import torch
print(f"PyTorch ready ({perf_counter() - _startup:.2f} s)", flush=True)
from IPython.display import Markdown, display

REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'main.py').is_file() and (p / 'tva').is_dir()
)
print("Loading TVA tokenizer classes...", flush=True)
from tva.tokenizer_factory import get_tokenizer
from tva.decoder_ctc import BestPath

MODELS = {
    'ED Bigram': ('handwriting_bigram_greedy', 'artifacts/tokenizers/ed/handwriting/ed_iam_handwriting_bigram_greedy_v1.json'),
    'ED BPE': ('handwriting_bpe', 'artifacts/tokenizers/ed/handwriting/ed_iam_handwriting_bpe_v1.json'),
    'ED Unigram': ('handwriting_unigram', 'artifacts/tokenizers/ed/handwriting/ed_iam_handwriting_unigram_v1.json')
}


def table(rows):
    if not rows:
        print('(no tokens)')
        return
    columns = list(rows[0])
    def clean(value):
        return str(value).replace('|', chr(92) + '|').replace(chr(10), ' ')
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    lines += ['| ' + ' | '.join(clean(row[column]) for column in columns) + ' |' for row in rows]
    display(Markdown(chr(10).join(lines)))


def tokens_from_ids(tokenizer, ids):
    mapping = tokenizer.idx_token
    if any(type(i) is not int or i not in mapping for i in ids):
        raise ValueError(f'IDs must be integers from 0 to {tokenizer.size - 1}.')
    return [mapping[i] for i in ids]


def scores_from_frame_ids(ids, size):
    if not ids:
        raise ValueError('A prediction needs at least one frame; use [0] for an all-blank path.')
    path = torch.tensor(ids, dtype=torch.long)
    scores = torch.full((len(ids), size), -8.0)
    scores.scatter_(1, path[:, None], 8.0)
    return scores


loaded = {}
artifact_hashes = {}
for name, (key, relative_path) in MODELS.items():
    artifact = REPO_ROOT / relative_path
    tokenizer = get_tokenizer(key)
    print(f"Loading {name}...", flush=True)
    tokenizer.load(artifact)
    print(f"{name} ready", flush=True)
    assert tokenizer.vocab[''] == 0
    loaded[name] = tokenizer
    artifact_hashes[name] = hashlib.sha256(artifact.read_bytes()).hexdigest()
table([{'name': name, 'classes including blank': t.size, 'blank ID': t.vocab['']} for name, t in loaded.items()])
print(f"Startup complete: {len(loaded)} tokenizers in {perf_counter() - _startup:.2f} s", flush=True)


Starting tokenizer inspection
Python: /home/artellisys/miniconda3/envs/tva/bin/python
Importing PyTorch...
PyTorch ready (1.16 s)
Loading TVA tokenizer classes...
Loading ED Bigram...


2026-10-17 21:58:07.713 | INFO     | tva.handwriting_tokenizers:load:363 - GreedyHandwritingBigramTokenizer is loaded from /home/artellisys/TVA/artifacts/tokenizers/ed/handwriting/ed_iam_handwriting_bigram_greedy_v1.json.


ED Bigram ready
Loading ED BPE...


2026-10-17 21:58:07.719 | INFO     | tva.handwriting_tokenizers:load:498 - HandwritingBPETokenizer is loaded from /home/artellisys/TVA/artifacts/tokenizers/ed/handwriting/ed_iam_handwriting_bpe_v1.json.


ED BPE ready
Loading ED Unigram...


2026-10-17 21:58:07.725 | INFO     | tva.handwriting_tokenizers:load:498 - HandwritingUnigramTokenizer is loaded from /home/artellisys/TVA/artifacts/tokenizers/ed/handwriting/ed_iam_handwriting_unigram_v1.json.


ED Unigram ready


| name | classes including blank | blank ID |
| --- | --- | --- |
| ED Bigram | 224 | 0 |
| ED BPE | 224 | 0 |
| ED Unigram | 224 | 0 |

Startup complete: 3 tokenizers in 1.20 s


## 1. Enter text and inspect encoding

Choose `ED Bigram`, `ED BPE`, `ED Unigram`, or `OnHW Bigram (historical)`. Spaces are displayed with `repr()` so they remain visible. Character offsets refer to NFC-normalized text.

ED Bigram takes an available pair greedily from left to right. BPE applies its learned merge order. Unigram uses the highest-scoring segmentation under its frozen token scores. The historical OnHW Bigram uses a maximum-total-utility segmentation. A Bigram's recorded utility is training evidence metadata; the **ED greedy encoder does not optimize that utility at runtime**.


In [3]:
# EDIT THESE TWO VALUES, then rerun this and the following cells.
TEXT = 'bookkeeper'
TOKENIZER_NAME = 'ED Bigram'

tokenizer = loaded[TOKENIZER_NAME]
normalized_text = unicodedata.normalize('NFC', TEXT)
missing = sorted(set(normalized_text) - set(tokenizer.vocab))
if missing:
    raise ValueError(f'{TOKENIZER_NAME} does not support {missing!r}. Use text from its alphabet; ED supports English spaces, digits and punctuation.')
segmentation = tokenizer.segment(TEXT)
encoded = tokenizer.encode(TEXT)

In [5]:
decoded = tokenizer.decode(encoded)
pieces = tokens_from_ids(tokenizer, encoded)
assert pieces == segmentation['tokens']
assert decoded == normalized_text and 0 not in encoded
assert BestPath(tokenizer).decode(torch.tensor(encoded, dtype=torch.long), label=True) == decoded
print('Input:       ', repr(TEXT))
print('Normalized:  ', repr(normalized_text))
print('Token pieces:', pieces)
print('Token IDs:   ', encoded)
print('Decoded:     ', repr(decoded))
print('Round trip:  ', decoded == normalized_text)
table([
    {'position': i, 'character span': f"{s['start']}:{s['end']}",
     'token': repr(s['token']), 'ID': token_id,
     'kind': s.get('kind', 'piece' if len(s['token']) > 1 else 'single'),
     'recorded utility': round(s['utility'], 6) if 'utility' in s else '—'}
    for i, (s, token_id) in enumerate(zip(segmentation['segments'], encoded))
])

Input:        'bookkeeper'
Normalized:   'bookkeeper'
Token pieces: ['bo', 'o', 'k', 'ke', 'e', 'pe', 'r']
Token IDs:    [122, 67, 63, 173, 57, 196, 70]
Decoded:      'bookkeeper'
Round trip:   True


| position | character span | token | ID | kind | recorded utility |
| --- | --- | --- | --- | --- | --- |
| 0 | 0:2 | 'bo' | 122 | handwriting-bigram | 0.562712 |
| 1 | 2:3 | 'o' | 67 | single | 0.0 |
| 2 | 3:4 | 'k' | 63 | single | 0.0 |
| 3 | 4:6 | 'ke' | 173 | handwriting-bigram | 0.666667 |
| 4 | 6:7 | 'e' | 57 | single | 0.0 |
| 5 | 7:9 | 'pe' | 196 | handwriting-bigram | 0.566013 |
| 6 | 9:10 | 'r' | 70 | single | 0.0 |

## 2. Compare the same text across tokenizers

IDs belong to each tokenizer's own vocabulary. An ID can mean different text in another model, even when both have 224 output classes. Decode a recognizer's output using the exact tokenizer artifact used to train that recognizer.

Unsupported characters are reported without deleting or replacing them. A successful round trip checks encoding consistency, not handwriting-recognition accuracy.


In [6]:
comparison_rows = []
for name, other in loaded.items():
    absent = sorted(set(normalized_text) - set(other.vocab))
    if absent:
        comparison_rows.append({'tokenizer': name, 'pieces': 'unsupported ' + repr(absent), 'IDs': '—', 'token count': '—', 'decoded': '—', 'round trip': '—'})
        continue
    ids = other.encode(TEXT)
    text = other.decode(ids)
    assert text == normalized_text and 0 not in ids
    comparison_rows.append({'tokenizer': name, 'pieces': repr(tokens_from_ids(other, ids)), 'IDs': repr(ids), 'token count': len(ids), 'decoded': repr(text), 'round trip': text == normalized_text})
table(comparison_rows)


| tokenizer | pieces | IDs | token count | decoded | round trip |
| --- | --- | --- | --- | --- | --- |
| ED Bigram | ['bo', 'o', 'k', 'ke', 'e', 'pe', 'r'] | [122, 67, 63, 173, 57, 196, 70] | 7 | 'bookkeeper' | True |
| ED BPE | ['b', 'oo', 'k', 'ke', 'e', 'p', 'er'] | [54, 177, 63, 156, 57, 68, 126] | 7 | 'bookkeeper' | True |
| ED Unigram | ['bo', 'o', 'k', 'ke', 'e', 'p', 'er'] | [109, 67, 63, 165, 57, 68, 136] | 7 | 'bookkeeper' | True |

## 3. Decode frame predictions with the real CTC decoder

A recognizer produces scores shaped **`[time steps, vocabulary size]`**, not an already segmented target string. TVA's `BestPath` takes the highest-scoring ID at each time step, collapses adjacent repeats, then joins the vocabulary pieces. Blank ID 0 maps to the empty string. Collapse happens **before** blank removal; a blank between two identical IDs allows a repeated token to survive.

With both editable variables below set to `None`, this cell builds **simulated scores** from the encoded text, repeating each token for two frames with blanks between tokens. This demonstrates decoding mechanics and is not a model recognition result.

To test your own predictions, set `FRAME_TOKEN_IDS` to a list of frame-level argmax IDs, or set `PRED_LOGITS` to a tensor of shape `[T, tokenizer.size]`. If you set both, logits take priority. Set `VALID_STEPS` when the tensor includes batch padding.


In [7]:
# OPTIONAL EDITS: use frame IDs OR a real [T, C] model-output tensor.
FRAME_TOKEN_IDS = None
PRED_LOGITS = None
VALID_STEPS = None

if PRED_LOGITS is not None:
    scores = torch.as_tensor(PRED_LOGITS).detach().cpu().to(torch.float32)
    origin = 'user-provided model scores'
else:
    if FRAME_TOKEN_IDS is None:
        frame_ids = [0]
        for token_id in encoded:
            frame_ids.extend([token_id, token_id, 0])
        origin = 'simulated scores from the encoded text'
    else:
        frame_ids = list(FRAME_TOKEN_IDS)
        origin = 'user-provided frame IDs converted to scores'
    tokens_from_ids(tokenizer, frame_ids)  # Reject invalid IDs before constructing scores.
    scores = scores_from_frame_ids(frame_ids, tokenizer.size)
if scores.ndim != 2 or scores.shape[1] != tokenizer.size or not len(scores):
    raise ValueError(f'Expected a nonempty [T, {tokenizer.size}] tensor; got {tuple(scores.shape)}.')
if VALID_STEPS is not None:
    if type(VALID_STEPS) is not int or not 1 <= VALID_STEPS <= len(scores):
        raise ValueError('VALID_STEPS must be an integer from 1 to T.')
    scores = scores[:VALID_STEPS]
raw_ids = torch.argmax(scores, dim=-1).tolist()
collapsed_ids = torch.unique_consecutive(torch.tensor(raw_ids)).tolist()
nonblank_ids = [i for i in collapsed_ids if i != 0]
predicted_text = BestPath(tokenizer).decode(scores)
assert predicted_text == tokenizer.decode(nonblank_ids)
print('Source:                 ', origin)
print('Frame-level IDs:        ', raw_ids)
print('After adjacent collapse:', collapsed_ids)
print('After removing blank:   ', nonblank_ids)
print('Final pieces:           ', tokens_from_ids(tokenizer, nonblank_ids))
print('BestPath prediction:    ', repr(predicted_text))
print('Direct decode(raw IDs): ', repr(tokenizer.decode(raw_ids)), '(does not perform CTC collapse)')
table([{'frame': i, 'winning ID': token_id, 'piece': '<CTC blank>' if token_id == 0 else repr(tokenizer.idx_token[token_id])} for i, token_id in enumerate(raw_ids)])
if FRAME_TOKEN_IDS is None and PRED_LOGITS is None and VALID_STEPS is None:
    assert predicted_text == normalized_text


Source:                  simulated scores from the encoded text
Frame-level IDs:         [0, 122, 122, 0, 67, 67, 0, 63, 63, 0, 173, 173, 0, 57, 57, 0, 196, 196, 0, 70, 70, 0]
After adjacent collapse: [0, 122, 0, 67, 0, 63, 0, 173, 0, 57, 0, 196, 0, 70, 0]
After removing blank:    [122, 67, 63, 173, 57, 196, 70]
Final pieces:            ['bo', 'o', 'k', 'ke', 'e', 'pe', 'r']
BestPath prediction:     'bookkeeper'
Direct decode(raw IDs):  'boboookkkekeeepeperr' (does not perform CTC collapse)


| frame | winning ID | piece |
| --- | --- | --- |
| 0 | 0 | <CTC blank> |
| 1 | 122 | 'bo' |
| 2 | 122 | 'bo' |
| 3 | 0 | <CTC blank> |
| 4 | 67 | 'o' |
| 5 | 67 | 'o' |
| 6 | 0 | <CTC blank> |
| 7 | 63 | 'k' |
| 8 | 63 | 'k' |
| 9 | 0 | <CTC blank> |
| 10 | 173 | 'ke' |
| 11 | 173 | 'ke' |
| 12 | 0 | <CTC blank> |
| 13 | 57 | 'e' |
| 14 | 57 | 'e' |
| 15 | 0 | <CTC blank> |
| 16 | 196 | 'pe' |
| 17 | 196 | 'pe' |
| 18 | 0 | <CTC blank> |
| 19 | 70 | 'r' |
| 20 | 70 | 'r' |
| 21 | 0 | <CTC blank> |

## 4. Repeated letters and CTC blanks

Two adjacent predictions of the same ID represent a sustained token and collapse to one. The same ID separated by a blank represents two tokens. Training target IDs are already a token sequence: decode them with `tokenizer.decode(ids)` or `BestPath(...).decode(ids_tensor, label=True)` so genuine repetitions are preserved.


In [8]:
checks = []
for name, other in loaded.items():
    letter_id = other.vocab['l']
    decoder = BestPath(other)
    cases = [
        ('all blank', [0, 0, 0], ''),
        ('same ID in adjacent frames', [letter_id, letter_id], 'l'),
        ('same ID separated by blank', [letter_id, 0, letter_id], 'll'),
        ('repeats around a blank', [0, letter_id, letter_id, 0, letter_id, letter_id, 0], 'll'),
    ]
    for explanation, ids, expected in cases:
        result = decoder.decode(scores_from_frame_ids(ids, other.size))
        assert result == expected
        checks.append({'tokenizer': name, 'case': explanation, 'frame IDs': repr(ids), 'CTC text': repr(result), 'passed': result == expected})
    target_ids = [letter_id, letter_id]
    assert other.decode(target_ids) == 'll'
    assert decoder.decode(torch.tensor(target_ids), label=True) == 'll'
table(checks)
print('All 16 CTC cases passed; repeated target tokens are preserved for all four tokenizers.')


| tokenizer | case | frame IDs | CTC text | passed |
| --- | --- | --- | --- | --- |
| ED Bigram | all blank | [0, 0, 0] | '' | True |
| ED Bigram | same ID in adjacent frames | [64, 64] | 'l' | True |
| ED Bigram | same ID separated by blank | [64, 0, 64] | 'll' | True |
| ED Bigram | repeats around a blank | [0, 64, 64, 0, 64, 64, 0] | 'll' | True |
| ED BPE | all blank | [0, 0, 0] | '' | True |
| ED BPE | same ID in adjacent frames | [64, 64] | 'l' | True |
| ED BPE | same ID separated by blank | [64, 0, 64] | 'll' | True |
| ED BPE | repeats around a blank | [0, 64, 64, 0, 64, 64, 0] | 'll' | True |
| ED Unigram | all blank | [0, 0, 0] | '' | True |
| ED Unigram | same ID in adjacent frames | [64, 64] | 'l' | True |
| ED Unigram | same ID separated by blank | [64, 0, 64] | 'll' | True |
| ED Unigram | repeats around a blank | [0, 64, 64, 0, 64, 64, 0] | 'll' | True |

All 16 CTC cases passed; repeated target tokens are preserved for all four tokenizers.


## 5. Look up tokens and decode your own target IDs

Use this cell for already-collapsed token sequences such as encoded labels. Raw frame predictions belong in section 3. Unknown characters or IDs raise an error instead of being silently removed.


In [9]:
TOKEN_TO_LOOK_UP = 'th'
TARGET_IDS_TO_DECODE = list(encoded)  # Replace with valid target/token IDs if desired.

print('Selected vocabulary:', TOKENIZER_NAME)
print('Lookup:', repr(TOKEN_TO_LOOK_UP), '->', tokenizer.vocab.get(TOKEN_TO_LOOK_UP, 'not in this vocabulary'))
print('IDs:   ', TARGET_IDS_TO_DECODE)
print('Pieces:', tokens_from_ids(tokenizer, TARGET_IDS_TO_DECODE))
print('Text:  ', repr(tokenizer.decode(TARGET_IDS_TO_DECODE)))
for name, (_, relative_path) in MODELS.items():
    assert hashlib.sha256((REPO_ROOT / relative_path).read_bytes()).hexdigest() == artifact_hashes[name]
print('Frozen tokenizer files unchanged.')


Selected vocabulary: ED Bigram
Lookup: 'th' -> 208
IDs:    [122, 67, 63, 173, 57, 196, 70]
Pieces: ['bo', 'o', 'k', 'ke', 'e', 'pe', 'r']
Text:   'bookkeeper'
Frozen tokenizer files unchanged.


### Using actual TVA recognizer output

During validation, TVA uses the matching tokenizer and trims padded frames before decoding, as in this example (the variables come from an existing model inference call):

```python
out = model(x)  # [batch, time, vocabulary size]
valid_steps = int(len_x[0]) // model.ratio_ds
PRED_LOGITS = out[0, :valid_steps].detach().cpu()
prediction = BestPath(tokenizer).decode(PRED_LOGITS)
```

You can paste that tensor into section 3. The model's vocabulary dimension must match `tokenizer.size`; the tokenizer artifact must also be the one used for that checkpoint. Saved `train_preds_*.json` files already contain decoded strings, so they do not contain the original frame logits or token IDs.
